# 11 — Deployment Report: US Project 05 Final Portfolio

A **deployment report**, not a code-validation notebook: it answers whether an
**already-built strategy is deployable**, not whether any code is correct.

The candidate is the Project 05 final US portfolio
(`weighted_multi_strategy_with_strategy_and_portfolio_smooth_dd`). This notebook
is a thin orchestrator: it loads the saved return series and daily weight panel
and hands them to `run_deployment_validation`. No strategy is reconstructed and
`run_market_robustness` is not used here.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/rawls/quant-lab


In [2]:
import pandas as pd

from src.analysis.deployment import (
    run_deployment_validation,
    select_best_per_group,
    pivot_metric_table,
)
from src.analysis.deployment_stress import (
    capacity_analysis,
    transaction_cost_drag,
    operational_stress_tests,
)
from src.analysis.liquidity import (
    load_price_volume,
    daily_dollar_volume,
    average_daily_volume,
    capacity_ceiling,
)
from src.analysis.turnover import (
    summarize_turnover,
    rolling_turnover_summary,
    turnover_spikes,
)

## 1. Load the deployment candidate

Two saved artifacts from `exp_005_risk_engine_final/`:

* `final_weighted_multi_strategy_portfolio_dd.csv` — the portfolio return /
  equity series (after both strategy- and portfolio-level drawdown overlays).
* `final_daily_weights.csv` — the daily asset-level weight panel
  (`Date, Ticker, Weight`) carrying the real trading dates.


In [3]:
EXP_DIR = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_final"

ret_df = pd.read_csv(EXP_DIR / "final_weighted_multi_strategy_portfolio_dd.csv")
weights_long = pd.read_csv(EXP_DIR / "final_daily_weights.csv", parse_dates=["Date"])

print("return series:", ret_df.shape, list(ret_df.columns))
print("weight panel :", weights_long.shape, list(weights_long.columns))

return series: (2558, 3) ['portfolio_return', 'equity_curve', 'portfolio_dd_exposure']
weight panel : (33254, 3) ['Date', 'Ticker', 'Weight']


## 2. Build `returns`, `weights`, `equity`

The return CSV is positional (no date column); the weight panel carries the
dates. Both have the same 2,558 trading days in the same order, so the sorted
weight-panel dates index the return and equity series.


In [4]:
# Wide weight matrix: dates x tickers
weights = (
    weights_long
    .pivot(index="Date", columns="Ticker", values="Weight")
    .sort_index()
    .fillna(0.0)
)

# Attach the dated index to the positional return / equity series
dates = weights.index
assert len(dates) == len(ret_df), (len(dates), len(ret_df))

returns = pd.Series(ret_df["portfolio_return"].to_numpy(), index=dates, name="returns")
equity = pd.Series(ret_df["equity_curve"].to_numpy(), index=dates, name="equity")

print("returns:", returns.shape, "| weights:", weights.shape, "| equity:", equity.shape)
print("date range:", dates.min().date(), "->", dates.max().date())
returns.head()

returns: (2558,) | weights: (2558, 20) | equity: (2558,)
date range: 2016-01-04 -> 2026-03-06


Date
2016-01-04    0.002917
2016-01-05    0.005973
2016-01-06   -0.002303
2016-01-07   -0.000840
2016-01-08   -0.008182
Name: returns, dtype: float64

## 3. Run the deployment-validation battery


In [5]:
TRANSACTION_COSTS = [0, 2, 5, 10, 20, 50]
REBALANCE_FREQUENCIES = [1, 2, 5, 10, "weekly"]

result = run_deployment_validation(
    returns=returns,
    weights=weights,
    equity=equity,
    transaction_costs=TRANSACTION_COSTS,
    rebalance_frequencies=REBALANCE_FREQUENCIES,
)

sorted(result.keys())

['rebalance_analysis',
 'rebalance_cost_grid',
 'regime_analysis',
 'rolling_metrics',
 'transaction_cost_stress',
 'turnover']

## 4. Turnover summary


In [6]:
turnover_summary = pd.Series(summarize_turnover(result["turnover"]), name="Turnover")
turnover_summary.to_frame()

,Turnover
mean,0.120094
median,0.100171
max,0.560404
p95,0.284440


## 5. Transaction-cost stress (as-deployed weight path)


In [7]:
cost_stress = result["transaction_cost_stress"]
cost_stress

,Cost bps,Sharpe,MDD,CAGR,Calmar,Mean Turnover
0,0,2.090113,-0.366949,0.377021,1.027446,0.120094
1,2,2.052352,-0.368698,0.368726,1.000074,0.120094
2,5,1.995681,-0.371312,0.356376,0.959773,0.120094
3,10,1.901156,-0.375646,0.336038,0.894561,0.120094
4,20,1.711843,-0.384223,0.296268,0.771083,0.120094
5,50,1.142168,-0.409260,0.183882,0.449303,0.120094


## 6. Rebalance analysis (turnover by cadence)


In [8]:
rebalance = result["rebalance_analysis"]
rebalance

,Rebalance Frequency,Mean Turnover,Median Turnover,Max Turnover,P95 Turnover
0,1,0.120094,0.100171,0.560404,0.284440
1,2,0.080067,0.000000,0.629559,0.301436
2,5,0.045911,0.000000,0.632745,0.302736
3,10,0.028381,0.000000,0.683104,0.261718
4,weekly,0.046753,0.000000,0.694892,0.302940


## 7. Best rebalance frequency by cost

From the `(rebalance frequency x cost)` grid, pick the frequency with the
highest net Sharpe at each cost level.


In [9]:
grid = result["rebalance_cost_grid"]

best_rebalance_by_cost = select_best_per_group(
    grid, group_cols=["Cost bps"], score_col="Sharpe", maximize=True
)
best_rebalance_by_cost[["Cost bps", "Rebalance Frequency", "Sharpe", "CAGR", "MDD", "Mean Turnover"]]

,Cost bps,Rebalance Frequency,Sharpe,CAGR,MDD,Mean Turnover
0,0,1,2.090113,0.377021,-0.366949,0.120094
1,2,10,2.081099,0.375054,-0.367448,0.028381
2,5,10,2.067565,0.372110,-0.368196,0.028381
3,10,10,2.044972,0.367216,-0.369440,0.028381
4,20,10,1.999660,0.357477,-0.371922,0.028381
5,50,10,1.862818,0.328654,-0.379311,0.028381


## 8. Deployment decision pivot

Net Sharpe across the full `rebalance frequency x cost` grid — the decision
matrix for choosing a deployment cadence under a given cost assumption.


In [10]:
decision_pivot = pivot_metric_table(
    grid, index="Rebalance Frequency", columns="Cost bps", value_col="Sharpe"
)
decision_pivot

Cost bps,0,2,5,10,20,50
Rebalance Frequency,,,,,,
1,2.090113,2.052352,1.995681,1.901156,1.711843,1.142168
10,2.090113,2.081099,2.067565,2.044972,1.999660,1.862818
2,2.090113,2.064904,2.027068,1.963945,1.837485,1.456804
5,2.090113,2.075594,2.053799,2.017426,1.944513,1.724681
weekly,2.090113,2.075323,2.053119,2.016067,1.941800,1.717922


## 9. Save US-specific outputs


In [11]:
results_dir = PROJECT_ROOT / "research/project_06_failure_analysis/results"
results_dir.mkdir(exist_ok=True)

turnover_summary.to_frame().to_csv(results_dir / "us_deployment_turnover_summary.csv")
cost_stress.to_csv(results_dir / "us_deployment_cost_stress.csv", index=False)
rebalance.to_csv(results_dir / "us_deployment_rebalance.csv", index=False)
best_rebalance_by_cost.to_csv(results_dir / "us_deployment_best_rebalance_by_cost.csv", index=False)
decision_pivot.to_csv(results_dir / "us_deployment_decision_pivot.csv")

print("Saved US deployment-validation outputs to", results_dir)

Saved US deployment-validation outputs to /Users/rawls/quant-lab/research/project_06_failure_analysis/results


## 10. Capacity analysis (real liquidity model)

How much capital can this book absorb before market impact eats the edge? This
section runs a **real liquidity model**, not a placeholder, following the
standard capacity chain end to end:

```
Daily Volume -> ADV -> Participation Rate -> Market Impact -> Slippage -> Net Returns
```

* **Daily Volume / ADV** — per-asset daily dollar volume (`Close x Volume`) for
  the 20 portfolio names is loaded from `data/raw/project_04_universe/` and
  smoothed into a 20-day trailing **average daily dollar volume (ADV)** panel.
* **Participation rate** — at each capital level, per-asset traded notional
  (`capital x |Δweight|`) divided by that asset's ADV.
* **Market impact** — the square-root law `coef * participation ** 0.5`
  (Almgren-style temporary impact), charged on the traded fraction.
* **Slippage -> Net returns** — the per-day asset-summed impact is a return drag
  subtracted from the gross series; `Sharpe`/`CAGR`/`MDD` are recomputed net.

Because real ADV now feeds the chain, net performance **degrades as capital
rises** (`ADV Provided == True`, `Mean Impact bps > 0`). We also report the
participation-cap **capacity ceiling**: the largest capital that keeps every
day's participation at or below 10% of ADV.


In [12]:
# Stage 0-1: Daily Volume -> ADV. Load real per-asset OHLCV for the 20 names
# and build a 20-day trailing average daily dollar-volume (ADV) panel.
close, volume = load_price_volume(
    list(weights.columns),
    data_dir=PROJECT_ROOT / "data/raw/project_04_universe",
)
adv = average_daily_volume(daily_dollar_volume(close, volume), window=20)

# Median end-of-sample ADV per name, as a sanity check (megacaps -> billions $).
print("ADV panel:", adv.shape,
      "| median latest ADV $: {:,.0f}".format(adv.reindex(dates).ffill().iloc[-1].median()))

# Stages 2-5: Participation -> Market Impact -> Slippage -> Net Returns, swept
# across capital levels. Real ADV now drives the square-root impact model.
CAPITAL_LEVELS = [10_000, 50_000, 100_000, 500_000, 1_000_000, 5_000_000]

capacity = capacity_analysis(
    returns=returns,
    weights=weights,
    capital_levels=CAPITAL_LEVELS,
    adv=adv,  # real ADV panel -> live market-impact / slippage
)
assert capacity["ADV Provided"].all(), "real liquidity model should be active"
capacity

ADV panel: (14170, 20) | median latest ADV $: 3,225,961,190


,Capital,ADV Provided,Mean Daily Traded $,Max Daily Traded $,Annual Traded $,Mean Participation,Mean Impact bps,Sharpe,CAGR,MDD
0,10000,True,1200.936473,5.604038e+03,3.026360e+05,1.001466e-07,0.140619,2.068082,0.372160,-0.367597
1,50000,True,6004.682364,2.802019e+04,1.513180e+06,5.007330e-07,0.314433,2.040838,0.366174,-0.368397
2,100000,True,12009.364728,5.604038e+04,3.026360e+06,1.001466e-06,0.444676,2.020415,0.361706,-0.368996
3,500000,True,60046.823640,2.802019e+05,1.513180e+07,5.007330e-06,0.994325,1.934143,0.343009,-0.371516
4,1000000,True,120093.647279,5.604038e+05,3.026360e+07,1.001466e-05,1.406188,1.869414,0.329167,-0.373398
5,5000000,True,600468.236397,2.802019e+06,1.513180e+08,5.007330e-05,3.144333,1.595536,0.272290,-0.381280


In [13]:
# Participation-cap capacity ceiling: the largest capital that keeps every day's
# participation at or below 10% of ADV (binding name per day, then summarised).
ceiling = capacity_ceiling(weights, adv, participation_cap=0.10)
capacity_ceiling_df = pd.DataFrame(
    {
        "participation_cap": [ceiling["participation_cap"]],
        "median_capital": [ceiling["median_capital"]],
        "p05_capital": [ceiling["p05_capital"]],
        "min_capital": [ceiling["min_capital"]],
    }
)
print("Capacity ceiling @ 10% participation:")
print("  median day : ${:,.0f}".format(ceiling["median_capital"]))
print("  5th pct day: ${:,.0f}".format(ceiling["p05_capital"]))
print("  worst day  : ${:,.0f}".format(ceiling["min_capital"]))
capacity_ceiling_df

Capacity ceiling @ 10% participation:
  median day : $1,932,241,564
  5th pct day: $509,181,617
  worst day  : $243,993,634


,participation_cap,median_capital,p05_capital,min_capital
0,0.1,1.932242e+09,5.091816e+08,2.439936e+08


In [14]:
capacity.to_csv(results_dir / "us_deployment_capacity_estimates.csv", index=False)
capacity_ceiling_df.to_csv(results_dir / "us_deployment_capacity_ceiling.csv", index=False)
print("Saved us_deployment_capacity_estimates.csv and us_deployment_capacity_ceiling.csv")
print("(real liquidity model: Daily Volume -> ADV -> Participation -> Impact -> Slippage)")

Saved us_deployment_capacity_estimates.csv and us_deployment_capacity_ceiling.csv
(real liquidity model: Daily Volume -> ADV -> Participation -> Impact -> Slippage)


## 11. Rolling turnover

Rolling 63-day (quarter) and 252-day (year) turnover — mean / max / p95 — plus
the dates where daily turnover spikes above a trailing 3-sigma threshold.


In [15]:
turnover = result["turnover"]

rolling_turnover = rolling_turnover_summary(turnover, windows=(63, 252))
rolling_turnover.dropna().tail()

,roll63_mean,roll63_max,roll63_p95,roll252_mean,roll252_max,roll252_p95
Date,,,,,,
2026-03-02,0.030083,0.072428,0.061562,0.047031,0.158392,0.104548
2026-03-03,0.029282,0.072428,0.060722,0.046773,0.158392,0.104548
2026-03-04,0.028843,0.072428,0.060722,0.046488,0.158392,0.104548
2026-03-05,0.028532,0.072428,0.060722,0.046511,0.158392,0.104548
2026-03-06,0.028205,0.072428,0.060722,0.046442,0.158392,0.104548


In [16]:
spikes = turnover_spikes(turnover, window=63, n_sigma=3.0)
print(f"{len(spikes)} turnover spikes (> trailing 63d mean + 3 sigma)")
spikes.head(10)

37 turnover spikes (> trailing 63d mean + 3 sigma)


,Date,turnover,roll_mean,roll_std,threshold,z_score
0,2019-06-20,0.236631,0.072281,0.025749,0.149528,6.382800
1,2025-01-27,0.238054,0.060658,0.031374,0.154780,5.654231
2,2018-08-14,0.316041,0.104756,0.044778,0.239091,4.718443
3,2023-10-13,0.499674,0.117752,0.081178,0.361286,4.704743
4,2020-05-18,0.549792,0.162090,0.085259,0.417869,4.547319
5,2023-07-14,0.286215,0.092748,0.043545,0.223383,4.442910
6,2016-06-01,0.436281,0.150837,0.065504,0.347348,4.357672
7,2022-03-07,0.231594,0.082090,0.035415,0.188336,4.221471
8,2019-06-26,0.255266,0.081894,0.042110,0.208222,4.117180
9,2018-08-28,0.376068,0.123801,0.061714,0.308944,4.087653


In [17]:
rolling_turnover.to_csv(results_dir / "us_deployment_rolling_turnover_summary.csv")
spikes.to_csv(results_dir / "us_deployment_rolling_turnover_spikes.csv", index=False)
print("Saved us_deployment_rolling_turnover_summary.csv and us_deployment_rolling_turnover_spikes.csv")

Saved us_deployment_rolling_turnover_summary.csv and us_deployment_rolling_turnover_spikes.csv


## 12. Rolling transaction-cost drag

Cumulative cost drag through time for each cost level, the resulting net equity
curves, and the final per-cost CAGR / Sharpe / MDD comparison (the summary
reuses the same `transaction_cost_stress` engine as section 5).


In [18]:
drag = transaction_cost_drag(returns, turnover, TRANSACTION_COSTS)

cost_drag_summary = drag["summary"]
cost_drag_summary

,Cost bps,Sharpe,MDD,CAGR,Calmar,Mean Turnover
0,0,2.090113,-0.366949,0.377021,1.027446,0.120094
1,2,2.052352,-0.368698,0.368726,1.000074,0.120094
2,5,1.995681,-0.371312,0.356376,0.959773,0.120094
3,10,1.901156,-0.375646,0.336038,0.894561,0.120094
4,20,1.711843,-0.384223,0.296268,0.771083,0.120094
5,50,1.142168,-0.409260,0.183882,0.449303,0.120094


In [19]:
# Net equity and cumulative drag through time (final row = end-of-sample)
net_equity = drag["net_equity"]
cumulative_drag = drag["cumulative_drag"]
pd.DataFrame({
    "Final Net Equity": net_equity.iloc[-1],
    "Final Cumulative Drag": cumulative_drag.iloc[-1],
})

,Final Net Equity,Final Cumulative Drag
Cost bps,,
0,25.725076,0.000000
2,24.194654,1.530422
5,22.067938,3.657138
10,18.930446,6.794630
20,13.929676,11.795400
50,5.548011,20.177065


In [20]:
cost_drag_summary.to_csv(results_dir / "us_deployment_cost_drag_summary.csv", index=False)
net_equity.to_csv(results_dir / "us_deployment_cost_drag_net_equity.csv")
cumulative_drag.to_csv(results_dir / "us_deployment_cost_drag_cumulative.csv")
print("Saved us_deployment_cost_drag_summary.csv, _net_equity.csv and _cumulative.csv")

Saved us_deployment_cost_drag_summary.csv, _net_equity.csv and _cumulative.csv


## 13. Operational stress tests

Execution-frictions battery at a 10 bps base cost: missed rebalances, a one-day
execution delay, doubled costs in high-volatility periods, and liquidity shocks
(2x / 3x cost) during the worst drawdown days. Gross returns are held fixed, so
each scenario isolates its **turnover / transaction-cost** impact (the same
deployment approximation used throughout this report).


In [21]:
operational_stress = operational_stress_tests(
    returns=returns,
    weights=weights,
    equity=equity,
    base_cost_bps=10.0,
    missed_rebalance_n=4,
    execution_delay_days=1,
    highvol_cost_multiplier=2.0,
    liquidity_shock_multipliers=(2.0, 3.0),
)
operational_stress

,Scenario,Sharpe,CAGR,MDD,Mean Turnover,Total Cost Drag
0,Baseline,1.901156,0.336038,-0.375646,0.120094,0.307200
1,Missed rebalance (every 4th),1.932605,0.342783,-0.374311,0.100073,0.255987
2,Execution delay (1d),1.900057,0.335961,-0.375718,0.120282,0.307681
3,High-vol 2x cost,1.872825,0.329931,-0.375756,0.120094,0.353822
4,Liquidity shock 2x (worst DD),1.891594,0.334043,-0.378352,0.120094,0.322365
5,Liquidity shock 3x (worst DD),1.882027,0.332050,-0.381048,0.120094,0.337531


In [22]:
operational_stress.to_csv(results_dir / "us_deployment_operational_stress_tests.csv", index=False)
print("Saved us_deployment_operational_stress_tests.csv")

Saved us_deployment_operational_stress_tests.csv
